## Launching model on dataset and collecting output

### Imports

In [21]:
from datasets import load_dataset, concatenate_datasets

from tqdm import tqdm

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from enum import Enum

import torch
import sys

sys.path.append("../services/")
from index import Index

torch.random.manual_seed(42)


### Preparation

In [22]:
ITERATIONS_COUNT = 684

In [23]:
PossibleOutput = Enum(
    "PossibleOutput", {f"PossibleValue{i}": str(i) for i in range(4)}
)


class OutputTemplate(BaseModel):
    short_chain_of_thoughts: str = Field(description="Short chain of thoughts")
    answer: PossibleOutput = Field(  # type: ignore
        description="One number of answer variant",
    )


def retrieve_answer_token_index(tokens):
    for i in range(len(tokens) - 1, 0, -1):
        if tokens[i]["token"].isdigit():
            return i


def retrieve_reasoning_tokens_range(tokens, start = "<think>", end = "</think>"):
    start_index, end_index = -1, -1
    tmp_text = ""
    for i in range(len(tokens)):
        if start in tmp_text:
            start_index = i
            break
        tmp_text += tokens[i]["token"]

    tmp_text = ""

    for i in range(len(tokens) - 1, 0, -1):
        tmp_text = tokens[i]["token"] + tmp_text
        if end in tmp_text:
            end_index = i
            break

    return (start_index, end_index)

In [28]:
dataset = load_dataset("reaganjlee/truthful_qa_mc")
dataset = concatenate_datasets([
    dataset["train"], 
    dataset["validation"]
])

chat_llm = ChatOpenAI(
    model="qwen2.5:7b-instruct",
    api_key="",
    base_url="http://localhost:11434/v1",
    max_tokens=4096,
    logprobs=True,
    top_logprobs=20,
)

parser = PydanticOutputParser(pydantic_object=OutputTemplate)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert at answering multiple choice questions. 
            You will be given a question and options. Follow these rules strictly:

            1. Think step by step, provide chain of thoughts leading to the answer
            2. Select the correct option number (0 - 4)
            3. Return your response in this EXACT JSON format:

            {format_instructions}

            Do not include any additional text outside the JSON format.
            Your response must be valid JSON that can be parsed automatically.
            Answer following question:
            """,
        ),
        (
            "human",
            """
            Question: {input_question}
    
            Options:
            {input_options}
            """,
        ),
    ]
)

llm_chain = prompt | chat_llm

### Running model on dataset and collecting results


In [29]:
def filter_func(model_response):
    """Return True/False whether answer token distribution does have all 0-9 tokens"""
    log = model_response.response_metadata["logprobs"]["content"][
        retrieve_answer_token_index(
            model_response.response_metadata["logprobs"]["content"]
        )
    ]
    if (
        len(
            set([str(x) for x in range(4)]).intersection(
                set([x["token"] for x in log["top_logprobs"]])
            )
        )
        < 4
    ):
        return False
    return True

In [32]:
# launching model

ERROR_LIMIT = 1000

index = Index(f"../index_data/qwen2.5-7B_ThrutfulQA_launch_{ITERATIONS_COUNT}")
index.clear()

correct_count = 0
progress_bar = tqdm(
    enumerate(dataset.select(range(ITERATIONS_COUNT))),
    total=ITERATIONS_COUNT,
    desc="Scoring ThrutfulQA",
)

for iter, data in progress_bar:
    for error_count in range(ERROR_LIMIT + 1):
        try:
            response = llm_chain.invoke(
                {
                    "format_instructions": parser.get_format_instructions(),
                    "input_question": dataset[iter]["question"],
                    "input_options": [
                        f"{i}:  {x}"
                        for i, x in enumerate(dataset[iter]["choices"])
                    ],
                }
            )

            if parser.parse(response.content) and filter_func(response):
                result = {
                    "iteration": iter,
                    "text": response.content,
                }

                if parser.parse(response.content).answer.value == str(
                    data["label"]
                ):
                    correct_count += 1

                progress_bar.set_postfix(
                    {"Current accuracy": str(correct_count / (iter + 1))}
                )

                result = {
                    "iteration": iter,
                    "text": response.content,
                    "logprobs": response.response_metadata["logprobs"][
                        "content"
                    ],
                }

                index.save_data(result, iter, logging=False)
                break

        except Exception as e:
            if error_count == ERROR_LIMIT:
                print("Errors limit exceeded")
                print(e)

    # print(data["answer_index"] == extract_predicted_index(text[-30:]))
    # print(extract_predicted_index(text[-30:]))
    # print("\n\n")

Файл не существует: ../index_data/qwen2.5-7B_ThrutfulQA_launch_684_index.pkl
Файл не существует: ../index_data/qwen2.5-7B_ThrutfulQA_launch_684_data.pkl
Index is cleared successfully.


Scoring ThrutfulQA: 100%|██████████| 684/684 [50:19<00:00,  4.41s/it, Current accuracy=0.6038011695906432] 


In [33]:
print(len(index))

684
